# 가사 교정 모델 파인튜닝 (무료 실행용)이 노트북은 **관리자가 직접** 실행합니다. 앱은 학습을 하지 않고, 여기서 만든결과물(ZIP)을 검증한 뒤 업로드할 때만 사용합니다.무료 Colab(T4) 또는 Kaggle 세션 한 번에 끝나도록 맞춰져 있습니다. 유료 리소스는필요하지 않습니다.## 무엇을 학습하나요앱의 여러 인식 모델이 **모두 같은 실수를 한 페이지**가 있습니다. 이럴 때는서로 대조해도 고를 것이 없습니다. 이 모델은 그런 페이지에서 사람이 실제로고쳐 저장한 결과를 배워, 합의 결과(consensus)를 한 번 더 다듬습니다.입력은 `manifest.jsonl`의 모델별 응답 + 합의 결과, 목표는 사람이 저장한 최종JSON입니다. 직렬화 형식은 앱의 `serializeCorrectionInput()`과 **완전히 같아야**합니다. 형식이 어긋나면 모델은 자신 있게 틀린 답을 내놓고, 이후 어떤 검증도그것을 깔끔하게 걸러내지 못합니다.## 순서1. 의존성 설치 (고정 버전)2. 관리자 설정에서 내려받은 학습 ZIP 업로드 및 검증3. 입력/목표 직렬화4. 페이지 해시 기준 8:2 분할 (같은 곡이 학습/검증에 걸치지 않도록)5. QLoRA 파인튜닝6. 베이스라인 대비 점수 비교7. 기준 미달이면 **결과물을 만들지 않고 중단**8. 어댑터 병합 → ONNX → int8 → manifest → ZIP 내려받기

## 1. 의존성 (고정 버전)버전을 고정해 두어야 몇 달 뒤 같은 노트북이 같은 결과를 냅니다.

In [ ]:
!pip -q install \    "transformers==4.57.1" \    "datasets==3.6.0" \    "peft==0.14.0" \    "accelerate==1.10.1" \    "optimum[onnxruntime]==1.24.0" \    "onnx==1.17.0" \    "onnxruntime==1.20.1" \    "sentencepiece==0.2.0" \    "bitsandbytes==0.45.0"import torchprint("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (CPU로도 동작하지만 매우 느립니다)")

## 2. 학습 ZIP 업로드관리자 설정 → 학습 자료 → **학습 자료 ZIP 내려받기**로 받은 파일을 올립니다.이 안에는 저작권이 있는 악보 이미지와 가사가 들어 있습니다. 공개 데이터셋으로업로드하거나 재배포하지 마세요.

In [ ]:
import hashlib, io, json, zipfilefrom pathlib import Pathtry:    from google.colab import files  # type: ignore    uploaded = files.upload()    ARCHIVE = io.BytesIO(next(iter(uploaded.values())))    ARCHIVE_BYTES = ARCHIVE.getvalue()except ImportError:    # Kaggle 등: 세션에 올려 둔 경로를 직접 지정합니다.    ARCHIVE_PATH = Path("lyrics-training.zip")    ARCHIVE_BYTES = ARCHIVE_PATH.read_bytes()DATASET_HASH = hashlib.sha256(ARCHIVE_BYTES).hexdigest()with zipfile.ZipFile(io.BytesIO(ARCHIVE_BYTES)) as archive:    names = set(archive.namelist())    assert "manifest.jsonl" in names, "manifest.jsonl 이 없습니다 — 학습 자료 ZIP이 맞는지 확인하세요."    records = [json.loads(line) for line in archive.read("manifest.jsonl").decode().splitlines() if line.strip()]print(f"학습 자료 {len(records)}건, dataset hash {DATASET_HASH[:16]}…")assert len(records) >= 50, "표본이 너무 적습니다. 최소 50건, 권장 100건 이상 검증한 뒤 다시 실행하세요."

## 3. 직렬화`serializeCorrectionInput()`(src/lib/learning/correctionModel.ts)과 **키 순서까지**같아야 합니다. 앱이 보내는 문자열과 학습에 쓴 문자열이 다르면 모델은 배운 적 없는입력을 받게 됩니다.

In [ ]:
def serialize_input(record):    consensus = record.get("consensus") or record.get("versions", [{}])[0] or {}    readings = []    for observation in record.get("observations", []):        score = observation.get("score")        if not score:            continue        attempt = observation.get("attempt", {})        readings.append({            "model": f"{attempt.get('engine','')}:{attempt.get('model','')}",            "title": score.get("title", "") or "",            "order": score.get("order", []) or [],            "sections": [                {"label": s.get("label", ""), "lines": s.get("lines", [])}                for s in score.get("sections", [])            ],        })    return json.dumps({        "task": "correct-korean-worship-lyrics",        "consensus": {            "title": consensus.get("title", "") or "",            "artist": consensus.get("artist", "") or "",            "key": consensus.get("key", "") or "",            "order": consensus.get("order", []) or [],            "sections": [                {"label": s.get("label", ""), "lines": s.get("lines", [])}                for s in consensus.get("sections", [])            ],        },        "readings": readings,    }, ensure_ascii=False, separators=(",", ":"))def serialize_target(record):    final = record.get("final") or (record.get("versions") or [{}])[-1]    return json.dumps({        "title": final.get("title", "") or "",        "artist": final.get("artist", "") or "",        "key": final.get("key", "") or "",        "order": final.get("order", []) or [],        "sections": [            {"label": s.get("label", ""), "lines": s.get("lines", [])}            for s in final.get("sections", [])        ],    }, ensure_ascii=False, separators=(",", ":"))examples = [    {"pageHash": r["pageHash"], "input": serialize_input(r), "target": serialize_target(r)}    for r in records]print(examples[0]["input"][:300])

## 4. 분할**페이지 해시 기준**으로 나눕니다. 같은 곡이 학습과 검증에 모두 들어가면 점수가실제보다 좋게 나오고, 그 점수를 믿고 배포하게 됩니다.

In [ ]:
SEED = 20260814import randompage_hashes = sorted({e["pageHash"] for e in examples})random.Random(SEED).shuffle(page_hashes)split_at = int(len(page_hashes) * 0.8)train_pages = set(page_hashes[:split_at])train = [e for e in examples if e["pageHash"] in train_pages]valid = [e for e in examples if e["pageHash"] not in train_pages]print(f"학습 {len(train)}건 / 검증 {len(valid)}건 (곡 단위로 분리)")

## 5. QLoRA 파인튜닝 (mT5-small, 무료 T4 기준)

In [ ]:
from datasets import Datasetfrom peft import LoraConfig, get_peft_modelfrom transformers import (    AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq,    EarlyStoppingCallback, Seq2SeqTrainer, Seq2SeqTrainingArguments,)BASE_MODEL = "google/mt5-small"MAX_INPUT, MAX_OUTPUT = 1024, 768tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["q", "v"]))model.print_trainable_parameters()def encode(batch):    encoded = tokenizer(batch["input"], max_length=MAX_INPUT, truncation=True)    encoded["labels"] = tokenizer(text_target=batch["target"], max_length=MAX_OUTPUT, truncation=True)["input_ids"]    return encodedtrain_ds = Dataset.from_list(train).map(encode, batched=True, remove_columns=["pageHash", "input", "target"])valid_ds = Dataset.from_list(valid).map(encode, batched=True, remove_columns=["pageHash", "input", "target"])trainer = Seq2SeqTrainer(    model=model,    args=Seq2SeqTrainingArguments(        output_dir="out",        seed=SEED,        per_device_train_batch_size=1,        gradient_accumulation_steps=8,        num_train_epochs=3,        learning_rate=2e-4,        eval_strategy="epoch",        save_strategy="epoch",        load_best_model_at_end=True,        logging_steps=10,        predict_with_generate=False,        fp16=torch.cuda.is_available(),        report_to=[],    ),    train_dataset=train_ds,    eval_dataset=valid_ds,    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],)trainer.train()

## 6. 점수 비교앱의 채점 코드(`src/lib/ai/recognitionScoring.ts`)를 그대로 옮겨 왔습니다. 앱과같은 방식으로 재야, 여기서 나온 수치가 실제 정확도와 같은 의미를 갖습니다.**베이스라인**은 "모델을 쓰지 않았을 때", 즉 합의 결과 그대로입니다.

In [ ]:
import re, unicodedatadef normalize_text(text):    return re.sub(r"[^0-9a-z\u3131-\u318e\uac00-\ud7a3]+", "", (text or "").lower())def levenshtein(a, b):    if a == b:        return 0    if not a or not b:        return len(a) or len(b)    prev = list(range(len(b) + 1))    for i, ca in enumerate(a, 1):        curr = [i]        for j, cb in enumerate(b, 1):            curr.append(min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + (ca != cb)))        prev = curr    return prev[-1]def text_similarity(a, b):    na, nb = normalize_text(a), normalize_text(b)    if not na and not nb:        return 1.0    longest = max(len(na), len(nb))    return 1.0 if longest == 0 else 1 - levenshtein(na, nb) / longestdef order_similarity(parsed, truth):    if not truth:        return 1.0 if not parsed else 0.0    a = [t.upper() for t in parsed]    b = [t.upper() for t in truth]    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]    for i in range(1, len(a) + 1):        for j in range(1, len(b) + 1):            dp[i][j] = dp[i - 1][j - 1] + 1 if a[i - 1] == b[j - 1] else max(dp[i - 1][j], dp[i][j - 1])    return dp[-1][-1] / max(len(a), len(b))def aliased(by_label, label):    want = label.strip().upper()    if want in by_label:        return by_label[want]    if not any(ch.isdigit() for ch in want):        return by_label.get(want + "1", "")    return by_label.get(re.sub(r"\d+$", "", want), "")def lyrics_similarity(parsed, truth):    by_label = {}    for section in parsed.get("sections", []):        key = section.get("label", "").strip().upper()        by_label[key] = (by_label.get(key, "") + " " + " ".join(section.get("lines", []))).strip()    scored = weight = 0.0    for section in truth.get("sections", []):        text = " ".join(section.get("lines", []))        w = len(normalize_text(text)) or 1        scored += text_similarity(aliased(by_label, section.get("label", "")), text) * w        weight += w    by_section = 1.0 if weight == 0 else scored / weight    whole = text_similarity(        " ".join(" ".join(s.get("lines", [])) for s in parsed.get("sections", [])),        " ".join(" ".join(s.get("lines", [])) for s in truth.get("sections", [])),    )    return 0.7 * by_section + 0.3 * wholedef score_one(parsed, truth):    title = text_similarity(parsed.get("title", ""), truth.get("title", ""))    order = order_similarity(parsed.get("order", []), truth.get("order", []))    lyrics = lyrics_similarity(parsed, truth)    return {"title": title, "order": order, "lyrics": lyrics,            "overall": 0.2 * title + 0.1 * order + 0.7 * lyrics}

In [ ]:
def generate(text):    batch = tokenizer(text, max_length=MAX_INPUT, truncation=True, return_tensors="pt").to(model.device)    output = model.generate(**batch, max_new_tokens=MAX_OUTPUT, num_beams=1)    return tokenizer.decode(output[0], skip_special_tokens=True)def safe_json(text):    try:        return json.loads(text)    except Exception:        return Nonebaseline_scores, tuned_scores = [], []for example in valid:    truth = json.loads(example["target"])    baseline = json.loads(example["input"])["consensus"]    baseline_scores.append(score_one(baseline, truth))    # 파싱조차 되지 않는 출력은 앱에서도 버려지므로, 그때는 합의 결과가 남습니다.    tuned_scores.append(score_one(safe_json(generate(example["input"])) or baseline, truth))def mean(rows, field):    return sum(row[field] for row in rows) / max(1, len(rows))BASELINE_OVERALL = mean(baseline_scores, "overall")TUNED_OVERALL = mean(tuned_scores, "overall")BASELINE_LYRICS = mean(baseline_scores, "lyrics")TUNED_LYRICS = mean(tuned_scores, "lyrics")print(f"overall  {BASELINE_OVERALL:.4f} → {TUNED_OVERALL:.4f}  ({TUNED_OVERALL - BASELINE_OVERALL:+.4f})")print(f"lyrics   {BASELINE_LYRICS:.4f} → {TUNED_LYRICS:.4f}")print(f"title    {mean(baseline_scores,'title'):.4f} → {mean(tuned_scores,'title'):.4f}")print(f"order    {mean(baseline_scores,'order'):.4f} → {mean(tuned_scores,'order'):.4f}")

## 7. 기준 미달이면 여기서 멈춥니다기준을 통과하지 못한 모델은 **결과물 자체를 만들지 않습니다**. 안 만들면 올릴 수도없고, 올릴 수 없으면 잘못 켜질 일도 없습니다.

In [ ]:
assert TUNED_OVERALL - BASELINE_OVERALL >= 0.01, (    f"전체 점수가 {TUNED_OVERALL - BASELINE_OVERALL:+.4f}만 올랐습니다. "    "매 페이지 추론 비용을 낼 만한 개선이 아니므로 결과물을 만들지 않습니다.")assert TUNED_LYRICS >= BASELINE_LYRICS, (    "가사 점수가 떨어졌습니다. 제목을 고치려고 가사를 망치는 모델이며, "    "슬라이드에 올라가는 것은 가사입니다.")print("기준 통과 — 결과물을 만듭니다.")

## 8. 병합 → ONNX → int8 → manifest → ZIP

In [ ]:
VERSION = "v1"  # 업로드마다 올려 주세요. URL의 일부가 됩니다.merged = model.merge_and_unload()merged.save_pretrained("merged")tokenizer.save_pretrained("merged")!optimum-cli export onnx --model merged --task text2text-generation-with-past onnx/!optimum-cli onnxruntime quantize --onnx_model onnx/ --avx2 -o onnx-int8/import shutilfrom pathlib import PathOUT = Path("artifact")if OUT.exists():    shutil.rmtree(OUT)(OUT / "onnx").mkdir(parents=True)for name in ["config.json", "tokenizer.json", "tokenizer_config.json", "generation_config.json",             "special_tokens_map.json", "spiece.model"]:    source = Path("merged") / name    if source.exists():        shutil.copy(source, OUT / name)for onnx_file in Path("onnx-int8").glob("*.onnx"):    shutil.copy(onnx_file, OUT / "onnx" / onnx_file.name)files = []for path in sorted(OUT.rglob("*")):    if not path.is_file():        continue    data = path.read_bytes()    files.append({        "path": str(path.relative_to(OUT)),        "size": len(data),        "sha256": hashlib.sha256(data).hexdigest(),    })manifest = {    "version": VERSION,    "baseModel": BASE_MODEL,    "datasetHash": DATASET_HASH,    "samples": len(examples),    "meanOverall": round(TUNED_OVERALL, 6),    "baselineOverall": round(BASELINE_OVERALL, 6),    "lyricsScore": round(TUNED_LYRICS, 6),    "baselineLyricsScore": round(BASELINE_LYRICS, 6),    "files": files,}(OUT / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2))archive_name = f"lyrics-corrector-{VERSION}.zip"with zipfile.ZipFile(archive_name, "w", zipfile.ZIP_DEFLATED) as out:    out.write(OUT / "manifest.json", "manifest.json")    for file in files:        out.write(OUT / file["path"], file["path"])print(f"{archive_name} — {sum(f['size'] for f in files)} bytes, {len(files)} files")try:    from google.colab import files as colab_files  # type: ignore    colab_files.download(archive_name)except ImportError:    print("이 파일을 내려받아 검증하세요.")

## 9. 업로드 전 검증 (로컬에서)```bashnode scripts/validate-correction-model.mjs lyrics-corrector-v1.zip```파일 해시, 점수 기준, 베이스 모델 허용 목록, 런타임 필수 파일을 다시 확인하고업로드 화면에 표시될 값을 그대로 출력합니다. 통과하면 관리자 설정 → 학습에서업로드하고 활성화하세요. 결과가 나쁘면 같은 화면에서 한 번에 되돌릴 수 있습니다.